In [78]:
# 1. 필요한 라이브러리 임포트
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt # 시각화 도구
import tensorflow as tf  # 딥러닝 라이브러리
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models  # 신경망 계층 및 모델 설계
from tensorflow.keras.optimizers import Adam, Nadam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0


In [79]:
# 폰트지정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 부호 깨짐 지정
plt.rcParams['axes.unicode_minus'] = False

# 숫자가 지수표현식으로 나올 때 지정
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# 2. 데이터 경로 및 기본 설정
base_dir = "images/vehicle_images"
train_dir = os.path.join(base_dir, "Train")
val_dir   = os.path.join(base_dir, "Validation")

IMG_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 1)
BATCH_SIZE = 32
NUM_CLASSES = 8
EPOCHS = 20

In [ ]:
# 3. 데이터 전처리 및 로딩

# ImageDataGenerator의 데이터 증강을 통해 훈련 시의 일반화 성능 증가
train_gen = ImageDataGenerator(
    rescale=1./255,                 # 픽셀 값을 0에서 1 사이로 정규화
    rotation_range=15,              # 이미지를 무작위로 15°  범위 내에서 회전
    width_shift_range=0.1,          # 이미지를 전체 너비의 최대 10% 내에서 수평 이동
    height_shift_range=0.1,         # 이미지를 전체 높이의 최대 10% 내에서 수직 이동
    zoom_range=0.1,                 # 이미지를 무작위로 최대 10% 확대/축소
    horizontal_flip=True            # 이미지를 무작위로 수평 뒤집기를 적용
)

val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)


Found 24000 images belonging to 8 classes.
Found 4800 images belonging to 8 classes.


In [32]:
# 4. 클래스 확인(차종 확인)
class_names = list(train_data.class_indices.keys())
class_names

['SUV', '버스', '세단', '승합', '이륜차', '트럭', '해치백', '화물']

In [ ]:
# 5. EarlyStopping 설정
# 콜백(Callback) 설정 및 폴더 생성
# ==========================================================

# 01. 모델 저장 폴더가 없으면 생성
if not os.path.exists('model'):
    os.makedirs('model')

# 02. EarlyStopping 설정
# 학습이 더 이상 개선되지 않으면 중단하여 시간을 절약합니다.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,               # 5번(Epoch) 동안 성능 향상이 없으면 중단
    restore_best_weights=True  # 중요: 멈춘 뒤, 메모리의 model 가중치를 '가장 좋았던 때'로 복구
)

In [38]:
# 6. 직적 만든 CNN 모델 간 성능 비교

# 01. CNN_Basic

def build_cnn_basic():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

    return model

In [39]:
# 02. CNN_Deep (더 깊은 구조)

def build_cnn_deep():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

    return model

In [40]:
# 03. CNN_With_Dropout (과적합 방지)

def build_cnn_dropout():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

    return model

In [ ]:
# 04. CNN 모델 정보 요약

models_dict = {
    "CNN_Basic": build_cnn_basic(),
    "CNN_Deep": build_cnn_deep(),
    "CNN_With_Dropout": build_cnn_dropout()
}

for name, model in models_dict.items():
    print(f"\n===== {name} =====")
    model.summary()



===== CNN_Basic =====


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,424 (91.20 MB)

 Trainable params: 23,908,424 (91.20 MB)

...

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,728 (84.86 MB)

 Trainable params: 22,246,728 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 05. CNN 훈련

models_dict = {
    "CNN_Basic": build_cnn_basic(),
    "CNN_Deep": build_cnn_deep(),
    "CNN_With_Dropout": build_cnn_dropout()
}

histories = {}

for model_name, model in models_dict.items():
    print(f"\n==============================")
    print(f" Training model: {model_name}")
    print(f"==============================")


    # ModelCheckpoint 설정
    # 가장 중요한 부분: val_loss가 가장 낮을 때만 파일로 저장합니다.


    # 각 모델마다 고유한 checkpoint 생성
    checkpoint_path = f'./model/best_vehicle_{model_name}.h5'
    print(f"DEBUG: checkpoint_path = '{checkpoint_path}'")

    checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,  # 저장할 경로
        monitor='val_loss',        # 기준: 검증 손실
        save_best_only=True,       # True: 이전보다 좋아졌을 때만 덮어쓰기 (최적 모델 저장)
        mode='min',                # loss는 낮을수록 좋으므로 min
        verbose=1                  # 저장될 때 로그 출력
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping]
    )

    histories[model_name] = history

    print(f"✓ {model_name} 훈련 완료! 저장 위치: {checkpoint_path}\n")

In [52]:
# 훈련한 cnn 모델 로드

from tensorflow.keras.models import load_model

model1 = load_model('model/best_vehicle_CNN_Basic.h5')
model2 = load_model('model/best_vehicle_CNN_Deep.h5')
model3 = load_model('model/best_vehicle_CNN_With_Dropout.h5')

In [ ]:
# 6. 모델별 Validation Loss 시각화(실제 모델 별 비교는 colab에서 실행)

plt.figure(figsize=(12, 6))

for model_name, history in histories.items():
    plt.plot(history.history['val_loss'], label=model_name, marker='o')

plt.title('Model Comparison - Validation Loss', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 결과: CNN_With_Dropout(val_loss: 0.4574) ,CNN_Basic(val_loss: 0.5206), CNN_Deep(val_loss: 0.5629)

#### 임의의 CNN모델 테스트 결과 CNN_With_Dropout 모델이 가장 성능이 우수

In [82]:
# 7. 최종 모델 비교를 위한 설정

# 01. CNN_With_Dropout (재정의)
def build_cnn_dropout_final():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [83]:
# 02. ResNet50 (전이 학습)
def build_resnet50():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=INPUT_SHAPE
    )
    
    # 처음에는 사전 학습된 가중치 동결
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [84]:
# 03. MobileNetV2 (전이 학습)
from tensorflow.keras.applications import MobileNetV2

def build_mobilenet():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=INPUT_SHAPE
    )
    
    # 처음에는 사전 학습된 가중치 동결
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [86]:
# 04. 최종 모델 비교 훈련

final_models_dict = {
    "CNN_With_Dropout": build_cnn_dropout_final(),
    "ResNet50": build_resnet50(),
    "MobileNetV2": build_mobilenet()
}

final_histories = {}

for model_name, model in final_models_dict.items():
    print(f"\n==============================")
    print(f" Training model: {model_name}")
    print(f"==============================")
    
    # 각 모델마다 고유한 checkpoint 생성
    checkpoint_path = f'./model/final_{model_name}.h5'
    
    checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    )
    
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping]
    )
    
    final_histories[model_name] = history
    
    print(f"✓ {model_name} 훈련 완료! 저장 위치: {checkpoint_path}\n")


 Training model: CNN_With_Dropout
Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4276 - loss: 1.4841
Epoch 1: val_loss improved from None to 0.68628, saving model to ./model/final_CNN_With_Dropout.h5


750/750 ━━━━━━━━━━━━━━━━━━━━ 814s 1s/step - accuracy: 0.5413 - loss: 1.1560 - val_accuracy: 0.7410 - val_loss: 0.6863
Epoch 2/20
338/750 ━━━━━━━━━━━━━━━━━━━━ 6:54 1s/step - accuracy: 0.6533 - loss: 0.8521

KeyboardInterrupt: 

In [ ]:
# 05. 각 모델별 Loss & Accuracy Curve 시각화

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

for idx, (model_name, history) in enumerate(final_histories.items()):
    # Loss Curve
    axes[idx, 0].plot(history.history['loss'], label='Train Loss', marker='o')
    axes[idx, 0].plot(history.history['val_loss'], label='Val Loss', marker='s')
    axes[idx, 0].set_title(f'{model_name} - Loss Curve', fontsize=14, fontweight='bold')
    axes[idx, 0].set_xlabel('Epoch', fontsize=11)
    axes[idx, 0].set_ylabel('Loss', fontsize=11)
    axes[idx, 0].legend()
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Accuracy Curve
    axes[idx, 1].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
    axes[idx, 1].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
    axes[idx, 1].set_title(f'{model_name} - Accuracy Curve', fontsize=14, fontweight='bold')
    axes[idx, 1].set_xlabel('Epoch', fontsize=11)
    axes[idx, 1].set_ylabel('Accuracy', fontsize=11)
    axes[idx, 1].legend()
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 06. 전체 모델 Validation Loss 비교

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
for model_name, history in final_histories.items():
    plt.plot(history.history['val_loss'], label=model_name, marker='o', linewidth=2)

plt.title('All Models - Validation Loss Comparison', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# 전체 모델 Validation Accuracy 비교
plt.subplot(1, 2, 2)
for model_name, history in final_histories.items():
    plt.plot(history.history['val_accuracy'], label=model_name, marker='s', linewidth=2)

plt.title('All Models - Validation Accuracy Comparison', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 07. 최종 성능 비교 바 차트

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(final_histories.keys())
best_val_losses = [min(final_histories[name].history['val_loss']) for name in model_names]
best_val_accs = [max(final_histories[name].history['val_accuracy']) for name in model_names]

# Best Validation Loss 비교
axes[0].bar(model_names, best_val_losses, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_title('Best Validation Loss Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# 값 표시
for i, v in enumerate(best_val_losses):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# Best Validation Accuracy 비교
axes[1].bar(model_names, best_val_accs, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Best Validation Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

# 값 표시
for i, v in enumerate(best_val_accs):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()